In [4]:
import pandas as pd
import numpy as np
from ISLP import load_data, confusion_table
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from ISLP.models import summarize, ModelSpec as MS
import statsmodels.api as sm
from sklearn.metrics import accuracy_score



# Question 5

## (a)

In [5]:
Default = load_data('Default')
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [ ]:
x = Default[['income', 'balance']]
x = sm.add_constant(x)
y = (Default['default']=='Yes').astype(int)


In [ ]:
model = sm.GLM(y, x, family=sm.families.Binomial())
results = model.fit()
summarize(results)

,coef,std err,z,P>|z|
const,-11.540500,0.435000,-26.544,0.0
income,0.000021,0.000005,4.174,0.0
balance,0.005600,0.000000,24.835,0.0


## (b)

In [ ]:
default_train, default_validate = train_test_split(Default, train_size=0.7, random_state=1)
x_train = default_train[['income', 'balance']]
x_train = sm.add_constant(x_train)
y_train = (default_train['default']=='Yes').astype(int)
model = sm.GLM(y_train, x_train, family=sm.families.Binomial())
results = model.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                default   No. Observations:                 7000
Model:                            GLM   Df Residuals:                     6997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -551.52
Date:                Wed, 22 Apr 2026   Deviance:                       1103.0
Time:                        09:58:43   Pearson chi2:                 4.14e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1333
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.8581      0.528    -22.477      0.000     -12.892     -10.824
income      2.223e-05      6e-06      3.707      0.000    1.05e-05     3.4e-05
balance        0.0059      0.000     21.078      0.000       0.005       0.006
==============================================================================
"""

In [ ]:
x_valid = default_validate[['income', 'balance']]
x_valid = sm.add_constant(x_valid)
y_valid = (default_validate['default'] == 'Yes').astype(int)
y_pred_prob = results.predict(x_valid)
y_pred = (y_pred_prob > 0.5).astype(int)




In [ ]:
confusion_table(y_pred, y_valid)

Truth,0,1
Predicted,,
0,2893,58
1,16,33


In [ ]:
print(f'{(1 - accuracy_score(y_pred, y_valid))*100:.3f} %')

2.467 %


## (c)

In [ ]:
def acc_score(test_):
    default_train, default_validate = train_test_split(Default, test_size=test_, random_state=1)
    x_train = default_train[['income', 'balance']]
    x_train = sm.add_constant(x_train)
    y_train = (default_train['default']=='Yes').astype(int)
    model = sm.GLM(y_train, x_train, family=sm.families.Binomial())
    results = model.fit()
    x_valid = default_validate[['income', 'balance']]
    x_valid = sm.add_constant(x_valid)
    y_valid = (default_validate['default'] == 'Yes').astype(int)
    y_pred_prob = results.predict(x_valid)
    y_pred = (y_pred_prob > 0.5).astype(int)
    return(f'{(1 - accuracy_score(y_pred, y_valid))*100:.3f}%')
    
    

In [ ]:
acc_score(0.5), acc_score(0.4), acc_score(0.3), acc_score(0.2), acc_score(0.1)

('2.500%', '2.525%', '2.467%', '2.700%', '3.000%')

## (d)

In [ ]:
default_train, default_validate = train_test_split(Default, test_size=0.3, random_state=1)
default_train['student'] = np.where(default_train['student'] == 'Yes', 1,0)
default_validate['student'] = np.where(default_validate['student'] == 'Yes', 1,0)
x_train = default_train[['income', 'balance', 'student']]
x_train = sm.add_constant(x_train)
y_train = (default_train['default']=='Yes').astype(int)
model = sm.GLM(y_train, x_train, family=sm.families.Binomial())
results = model.fit()
x_valid = default_validate[['income', 'balance', 'student']]
x_valid = sm.add_constant(x_valid)
y_valid = (default_validate['default'] == 'Yes').astype(int)
y_pred_prob = results.predict(x_valid)
y_pred = (y_pred_prob > 0.5).astype(int)
print(f'{(1 - accuracy_score(y_pred, y_valid))*100:.3f}%')

2.433%


using islp lib we can do as following:

In [ ]:
design = MS(['income', 'balance', 'student'])
x = design.fit_transform(Default)
y = Default['default'] == 'Yes'
x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.3, random_state=1)
model = sm.GLM(y_train, x_train, family=sm.families.Binomial())
results = model.fit()
pred_proba = results.predict(x_valid)
y_pred = np.where(pred_proba>0.5, 1, 0)
1- accuracy_score(y_pred, y_valid)



0.024333333333333318

# Question 6

## (a)

In [21]:
design = MS(['income', 'balance'])
x = design.fit_transform(Default)
y = Default['default'] == 'Yes'
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)
model = sm.GLM(y_train, x_train, family=sm.families.Binomial())
results = model.fit()
summarize(results)
results.bse



intercept    0.527555
income       0.000006
balance      0.000278
dtype: float64

## (b)

In [18]:
def boot_fn(df: pd.DataFrame, idx: np.ndarray):
    design = MS(['income', 'balance'])
    data = df.loc[idx]
    x = design.fit_transform(data)
    y = data['default'] == 'Yes'
    
    model = sm.GLM(y, x, family=sm.families.Binomial())
    results = model.fit()
    return results.params.loc[['income', 'balance']]
    

In [10]:
rng = np.random.default_rng(0)
boot_fn(Default, rng.choice(len(Default), 100, replace=True))

income     0.000037
balance    0.005290
dtype: float64

## (c)

In [19]:
def boot_se(func,
            df,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or df.shape[0]
    for _ in range(B):
        idx = rng.choice(len(df), n, replace=True)
        value = func(df, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_/B - (first_/B)**2)
    
    
    

In [20]:
SE_params = boot_se(boot_fn, Default, B=1000, seed=0)
SE_params

income     0.000005
balance    0.000230
dtype: float64

## (d)

In [22]:
print(results.bse.loc[['income', 'balance']])
print(SE_params)

income     0.000006
balance    0.000278
dtype: float64
income     0.000005
balance    0.000230
dtype: float64
